# DL · Encuentro 3 · Secuencias, atención y Transformer

Secuencias, atención y contexto

**Tu copia personal:** en Colab elige **Archivo → Guardar una copia en Drive**. Conserva esa copia para los tres temas del encuentro.

Trabaja en CPU. Estas prácticas no requieren contratar GPU, claves de API ni servicios de pago. La primera preparación puede descargar bibliotecas ausentes; las prácticas de detección descargan los pesos del modelo si no están disponibles.

## Tu recorrido

1. [El orden puede cambiar el significado](#scrollTo=tema-dl-secuencias)
2. [Relacionar elementos con contexto](#scrollTo=tema-dl-atencion)
3. [Atención más información de orden](#scrollTo=tema-dl-transformers)

Cada tema reúne **idea → caso explicado → preparación → ejecución → lectura y comprobación**. Puedes ejecutar todo el cuaderno o empezar por un tema: cada uno vuelve a preparar sus datos y herramientas. No mezcles las cifras de conjuntos diferentes.

Los datos inventados se identifican como simulaciones. Las fotos de detección son demostraciones y no incluyen una evaluación anotada de sus objetos. Las comprobaciones son voluntarias y sin nota.


## Tema 1 · Pronosticar una lectura con LSTM

**Lo que vas a poder hacer:** Construir un resumen de una secuencia que tenga en cuenta el orden.

Un resumen de sensores o ventas puede perder información si ignora qué ocurrió primero.

### Entiende la idea

Una secuencia es una lista en la que importa qué ocurrió primero y qué vino después. Recibir 10, luego 20 y luego 30 cajas cuenta una historia diferente de recibirlas en el orden inverso. Una regla recurrente conserva un pequeño resumen del pasado y lo actualiza con cada nueva lectura. A ese resumen lo llamamos estado. Primero veremos un estado calculado con una regla fija y sencilla. Una RNN es una red que aprende cómo actualizarlo. Una LSTM (Long Short-Term Memory; memoria de largo y corto plazo) es un tipo de RNN con mecanismos para conservar o descartar información durante ese recorrido.

### Un caso pequeño antes del programa

**Los mismos valores, dos recorridos**

Una operación registra cajas recibidas en tres horas: 10, 20 y 30. Comparamos ese orden con 30, 20 y 10. Para ilustrar memoria empezamos con estado 0 y calculamos estado nuevo = 0,5 × estado anterior + 0,5 × entrada. Los coeficientes son elegidos, no entrenados. El estado es un resumen ponderado de las cantidades, no el pronóstico de la siguiente hora.

| Paso | Entrada ascendente | Estado | Entrada descendente | Estado |
| --- | --- | --- | --- | --- |
| 1 | 10 | 5 | 30 | 15 |
| 2 | 20 | 12,5 | 20 | 17,5 |
| 3 | 30 | 21,25 | 10 | 13,75 |

1. En orden ascendente, el primer estado es 0,5 × 0 + 0,5 × 10 = 5.
2. El segundo es 0,5 × 5 + 0,5 × 20 = 12,5; el tercero es 21,25.
3. En orden inverso, los estados son 15, 17,5 y 13,75: los últimos valores influyen de manera diferente en este recorrido.

**Qué demuestra:** El promedio coincide, pero el estado final cambia porque la regla conserva parte del pasado.

### Ahora identifica los datos de esta práctica

Cada predicción usa una ventana de doce lecturas anteriores. Compara LSTM y último valor sobre los mismos instantes reservados.

El caso pequeño y la ejecución pueden usar conjuntos diferentes: interpreta las cifras con las unidades y filas declaradas en cada uno.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere ejecutar los temas anteriores. Si repites un tema, comienza desde su primera celda.

**Qué cambiar y qué conservar:** La regla de mitades del aula explica memoria con números fáciles. La LSTM sí aprende sus parámetros y usa otra serie; sus cifras no deben coincidir.


In [ ]:
# Preparación 1: instalar únicamente bibliotecas que falten.
import importlib.util, subprocess, sys
paquetes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
            'sklearn':'scikit-learn'}
for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                           '--index-url', 'https://download.pytorch.org/whl/cpu'])
print('Bibliotecas listas. No necesitas activar GPU.')

# Las extensiones se preparan solo cuando se abre su bloque.
def preparar_extension(modulo, paquete):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])


In [ ]:
# Preparación 2: herramientas compartidas.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
from sklearn.datasets import load_digits
from IPython.display import display

# Estas opciones mantienen fuera de la ruta principal el material de ampliación.
# Déjalas en False para los cuatro encuentros de DL.
EJECUTAR_REPASO_ML = False
EJECUTAR_TEXTO = False
EJECUTAR_GENERACION = False

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA); torch.set_num_threads(1)
AZUL, VERDE, GRIS = '#0075ad', '#198a87', '#9eb4c7'
plt.rcParams.update({'figure.figsize':(9,4.5), 'font.size':11,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.titleweight':'bold', 'axes.grid':False})
# En Colab puedes descargar los PNG desde el panel Archivos.
CARPETA = Path('figuras_utp'); CARPETA.mkdir(exist_ok=True)
def mostrar(nombre):
    plt.tight_layout()
    plt.savefig(CARPETA / (nombre + '.png'), dpi=150, bbox_inches='tight')
    plt.show()

def datos_fincas(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Humedad_pct':r.uniform(20,70,n),
                      'Temperatura_C':r.uniform(18,35,n),
                      'Lluvia_mm':r.uniform(0,100,n)})
    y = (2 + .07*X.Humedad_pct + .012*X.Lluvia_mm
         - .025*(X.Temperatura_C-24)**2 + r.normal(0,.3,n))
    return X, y.to_numpy()

def datos_cafe(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Uniformidad_color':r.uniform(0,1,n),
                      'Proporcion_manchas':r.uniform(0,1,n)})
    puntuacion = 2*X.Uniformidad_color-X.Proporcion_manchas-.5+r.normal(0,.15,n)
    return X, (puntuacion > 0).astype(int).to_numpy() # 1=sano, 0=defectuoso
print('Semilla:',SEMILLA,'· PyTorch:',torch.__version__)
print('Todas las cifras agrícolas son simuladas.')


### 11A · Pronosticar la siguiente lectura con LSTM

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


In [ ]:
# Una LSTM real para una secuencia pequeña y artificial.
r=np.random.default_rng(SEMILLA);tiempo=np.arange(240);serie=np.sin(tiempo*.12)+.12*r.normal(size=len(tiempo))
VENTANA=12;CORTE=180
entradas=np.array([serie[i-VENTANA:i] for i in range(VENTANA,len(serie))],dtype='float32')
objetivos=serie[VENTANA:].astype('float32');instante=tiempo[VENTANA:]
train=instante<CORTE;test=~train
xt=torch.tensor(entradas[train,:,None]);yt=torch.tensor(objetivos[train,None]);xe=torch.tensor(entradas[test,:,None])
class PredictorLSTM(nn.Module):
    def __init__(self):
        super().__init__();self.memoria=nn.LSTM(1,12,batch_first=True);self.salida=nn.Linear(12,1)
    def forward(self,x):
        secuencia,_=self.memoria(x);return self.salida(secuencia[:,-1])
torch.manual_seed(SEMILLA);lstm=PredictorLSTM();opt=torch.optim.Adam(lstm.parameters(),lr=.02)
for epoca in range(90):
    opt.zero_grad();error=nn.functional.mse_loss(lstm(xt),yt);error.backward();opt.step()
lstm.eval()
with torch.no_grad():p=lstm(xe).numpy().ravel()
baseline=entradas[test,-1]
print('MAE LSTM:',round(mean_absolute_error(objetivos[test],p),3),'· MAE último valor:',round(mean_absolute_error(objetivos[test],baseline),3))
plt.plot(instante[test],objetivos[test],label='Real simulado',color=GRIS);plt.plot(instante[test],p,label='LSTM',color=AZUL);plt.plot(instante[test],baseline,label='Último valor',color=VERDE,alpha=.6)
plt.legend();plt.xlabel('Tiempo');plt.ylabel('Lectura simulada (sin unidades)');plt.title('Pronóstico de un paso usando la ventana disponible');mostrar('11-lstm')
print('Se evalúa un paso hacia adelante; no es un pronóstico recursivo de toda la serie.')


### Después de ejecutar: compara en los mismos instantes

La línea real es la respuesta reservada; la LSTM y Último valor son dos propuestas para el mismo instante. Busca una zona de subida y otra de bajada y compara cuánto se separan. El MAE resume esas distancias. Repetir el último valor es una referencia que el modelo puede o no mejorar.


### Lee, comprueba y explica

Cada predicción usa una ventana de doce lecturas anteriores. Compara LSTM y último valor sobre los mismos instantes reservados.

**Prueba pequeña:** La regla de mitades del aula explica memoria con números fáciles. La LSTM sí aprende sus parámetros y usa otra serie; sus cifras no deben coincidir.

**Vuelve al caso:** Conserva el orden y registra cada actualización del estado. El estado resume lo leído hasta ese paso.

**Pregunta:** ¿El estado recurrente guarda una copia exacta de todo lo anterior?

**Respuesta razonada:** No. Es un resumen que depende de la regla y de sus parámetros. En la demostración es una mezcla numérica fija; una RNN aprende su transformación y una LSTM usa mecanismos adicionales para regular la memoria.

**Registra en tu copia:** qué representan las filas, qué valores cambiaste, qué observaste y qué límite tiene la conclusión. No es una entrega ni una calificación.

[Volver al tema del aula](https://kthyon.github.io/diplomado-utp-aula/#tema/dl-secuencias)


## Tema 2 · Mezclar dos registros con atención

**Lo que vas a poder hacer:** Construir una combinación ponderada de fuentes y rastrear su cálculo.

Permite explicar una combinación interna sin presentarla como comprensión humana o explicación causal.

### Entiende la idea

Atención es una operación que mezcla información dando diferente participación a cada fuente. Si mezclamos por mitades 10 y 20, obtenemos 15. Si damos más participación a 10, el resultado se acerca a 10. Esa participación es un peso: todos los pesos de esta mezcla suman uno, es decir, el 100 %. Para calcularlos se compara una consulta con las descripciones de las fuentes. La consulta se llama Q, las descripciones se llaman claves K y la información que mezclamos son los valores V. Primero seguiremos el reparto visible; las listas y multiplicaciones aparecen después, como detalle de cómo se calcula.

### Un caso pequeño antes del programa

**Una consulta, dos fuentes**

Dos registros ficticios de una operación aportan cantidades de 10 y 20 cajas. Para estudiar cómo se combinan, les damos claves K_A = [1,0] y K_B = [0,1]. La consulta Q y esas claves son vectores elegidos para el cálculo, no características aprendidas de pedidos reales. V_A = 10 y V_B = 20 son los valores que mezclaremos. El resultado es una representación ponderada, no una orden de compra ni un pronóstico validado.

| Consulta Q | Productos con A y B | Pesos aproximados | Salida |
| --- | --- | --- | --- |
| [1,0] | 1 y 0 | 0,6698 y 0,3302 | 13,302 |
| [0,1] | 0 y 1 | 0,3302 y 0,6698 | 16,698 |
| [0,0] | 0 y 0 | 0,5 y 0,5 | 15 |

1. Con Q = [1,0], Q·K_A = 1×1 + 0×0 = 1 y Q·K_B = 1×0 + 0×1 = 0. Dividimos ambos por √2, porque las claves tienen dos componentes: obtenemos aproximadamente 0,7071 y 0.
2. Softmax calcula el peso de A como exp(0,7071) / [exp(0,7071) + exp(0)] ≈ 0,6698. El peso de B es ≈ 0,3302; ambos suman 1.
3. Combina los valores: 0,6698 × 10 + 0,3302 × 20 ≈ 13,302, usando los pesos sin redondear.

**Qué demuestra:** La consulta cambia el reparto y la salida, aunque claves y valores permanezcan iguales.

### Ahora identifica los datos de esta práctica

Sigue A=10 y B=20 desde el reparto de pesos hasta su mezcla. Comprueba que cada reparto suma 1, es decir, 100 %.

El caso pequeño y la ejecución pueden usar conjuntos diferentes: interpreta las cifras con las unidades y filas declaradas en cada uno.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere ejecutar los temas anteriores. Si repites un tema, comienza desde su primera celda.

**Qué cambiar y qué conservar:** Con participaciones iguales, la mezcla debe ser 15. Un peso mayor para A la acerca a 10; un peso mayor para B, a 20.


In [ ]:
# Preparación 1: instalar únicamente bibliotecas que falten.
import importlib.util, subprocess, sys
paquetes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
            'sklearn':'scikit-learn'}
for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                           '--index-url', 'https://download.pytorch.org/whl/cpu'])
print('Bibliotecas listas. No necesitas activar GPU.')

# Las extensiones se preparan solo cuando se abre su bloque.
def preparar_extension(modulo, paquete):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])


In [ ]:
# Preparación 2: herramientas compartidas.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
from sklearn.datasets import load_digits
from IPython.display import display

# Estas opciones mantienen fuera de la ruta principal el material de ampliación.
# Déjalas en False para los cuatro encuentros de DL.
EJECUTAR_REPASO_ML = False
EJECUTAR_TEXTO = False
EJECUTAR_GENERACION = False

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA); torch.set_num_threads(1)
AZUL, VERDE, GRIS = '#0075ad', '#198a87', '#9eb4c7'
plt.rcParams.update({'figure.figsize':(9,4.5), 'font.size':11,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.titleweight':'bold', 'axes.grid':False})
# En Colab puedes descargar los PNG desde el panel Archivos.
CARPETA = Path('figuras_utp'); CARPETA.mkdir(exist_ok=True)
def mostrar(nombre):
    plt.tight_layout()
    plt.savefig(CARPETA / (nombre + '.png'), dpi=150, bbox_inches='tight')
    plt.show()

def datos_fincas(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Humedad_pct':r.uniform(20,70,n),
                      'Temperatura_C':r.uniform(18,35,n),
                      'Lluvia_mm':r.uniform(0,100,n)})
    y = (2 + .07*X.Humedad_pct + .012*X.Lluvia_mm
         - .025*(X.Temperatura_C-24)**2 + r.normal(0,.3,n))
    return X, y.to_numpy()

def datos_cafe(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Uniformidad_color':r.uniform(0,1,n),
                      'Proporcion_manchas':r.uniform(0,1,n)})
    puntuacion = 2*X.Uniformidad_color-X.Proporcion_manchas-.5+r.normal(0,.15,n)
    return X, (puntuacion > 0).astype(int).to_numpy() # 1=sano, 0=defectuoso
print('Semilla:',SEMILLA,'· PyTorch:',torch.__version__)
print('Todas las cifras agrícolas son simuladas.')


### Consulta, claves y valores son símbolos

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **Q, K y V** — Q, K y V: Consulta, clave y valor. Q identifica la consulta; K, las claves con las que se compara; V, la información que se mezcla. Son símbolos de cantidades, no tres siglas de algoritmos. La comparación Q–K determina pesos; la combinación de V produce la salida, sin demostrar causalidad. [Fuente](https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html)

- **Softmax** — Softmax: Transformación de puntuaciones en participaciones. Para cada puntuación calcula su exponencial y la divide entre la suma de exponenciales del grupo. Así obtiene valores positivos que suman uno cuando las entradas son finitas. La normalización se hace sobre un eje elegido; en atención, por las fuentes de cada consulta. [Fuente](https://docs.pytorch.org/docs/stable/generated/torch.nn.Softmax.html)

- **Transformer** — Transformer: Arquitectura de red basada en atención. Arquitectura que combina atención con otras transformaciones, conexiones residuales y normalización. Incorpora información de posición para distinguir órdenes en la tarea del cuaderno. No es una sigla, una sola operación ni necesariamente un chatbot. [Fuente](https://arxiv.org/abs/1706.03762)


### Lee una mezcla antes de leer sus símbolos

Si tomas la mitad de 10 y la mitad de 20, tienes 5+10=15. En la cuenta de atención, una consulta [1,0] da aproximadamente 67 % de participación a A y 33 % a B: la mezcla queda cerca de 13,3, más próxima a 10.

**Vector** significa lista ordenada de números. [1,0] tiene dos posiciones; no es una fracción. **Producto punto** multiplica números del mismo lugar y suma: [1,0] con [1,0] da 1×1 + 0×0 = 1. Se divide por la raíz de 2 porque las listas tienen dos posiciones. **Softmax** convierte esas comparaciones en participaciones positivas que suman 1.

**Tip:** la consulta [0,0] compara igual con A y B. Comprueba que su reparto sea 0,5 y 0,5 y que la salida sea 15. Atención calcula una representación; 13,3 no es una orden de compra ni una cantidad aprendida de ventas.

En el Transformer siguiente sí se ajustan parámetros usando ejemplos de una tarea distinta. Compara sus respuestas con las cuatro etiquetas que resolviste. **Etiqueta** es la respuesta correcta conocida; **predicción** es lo que calcula la red. Una puntuación alta de la red todavía puede equivocarse.


### 12A · Mezclar dos registros con atención

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


In [ ]:
# Dos registros como en el aula: mezclamos cantidades con pesos dados por la comparación.
fuentes=['A: 10','B: 20']
consultas=['Q=[1,0]','Q=[0,1]','Q=[0,0]']
Q=np.array([[1.,0.],[0.,1.],[0.,0.]])
K=np.array([[1.,0.],[0.,1.]])
V=np.array([[10.],[20.]])
puntuaciones=Q@K.T/np.sqrt(2)
exp=np.exp(puntuaciones-puntuaciones.max(axis=1,keepdims=True))
atencion=exp/exp.sum(axis=1,keepdims=True)
mezclas=atencion@V
assert np.allclose(atencion.sum(axis=1),1)
assert np.allclose(atencion[2],[.5,.5]) and np.isclose(mezclas[2,0],15)
for consulta,pesos,mezcla in zip(consultas,atencion,mezclas[:,0]):
    print(consulta,'· participación A/B:',np.round(pesos,4),'· mezcla:',round(float(mezcla),3))
fig,axs=plt.subplots(1,2,figsize=(10,4))
axs[0].imshow(atencion,cmap='Blues',vmin=0,vmax=1)
axs[0].set_xticks(range(2),fuentes);axs[0].set_yticks(range(3),consultas)
for i in range(3):
    for j in range(2):axs[0].text(j,i,f'{atencion[i,j]*100:.1f} %',ha='center',va='center',color='black')
axs[0].set(title='Cuánto participa cada fuente',xlabel='Fuente',ylabel='Consulta dada')
axs[1].bar(consultas,mezclas[:,0],color=[AZUL,VERDE,GRIS]);axs[1].set(title='Resultado de cada mezcla',ylabel='Valor mezclado',ylim=(0,22))
axs[1].axhline(10,ls=':',color='gray');axs[1].axhline(20,ls=':',color='gray')
mostrar('12-atencion')


### Después de ejecutar: comprueba el reparto

Cada fila debe sumar 100 %. Q=[0,0] produce partes iguales y mezcla 15. Q=[1,0] da más participación a A=10 y una salida cercana a 13,3. No se entrenó atención aquí: Q, K y V fueron elegidos para enseñar la operación.


### Lee, comprueba y explica

Sigue A=10 y B=20 desde el reparto de pesos hasta su mezcla. Comprueba que cada reparto suma 1, es decir, 100 %.

**Prueba pequeña:** Con participaciones iguales, la mezcla debe ser 15. Un peso mayor para A la acerca a 10; un peso mayor para B, a 20.

**Vuelve al caso:** Sigue consulta y claves → puntuaciones → pesos → suma ponderada de valores. Los pesos indican cuánto aporta cada valor a esa combinación.

**Pregunta:** ¿Qué papel cumplen Q, K, V y los pesos de atención?

**Respuesta razonada:** Q plantea la comparación; K describe cada fuente para compararla; V contiene lo que se combinará. Las comparaciones producen pesos, y estos ponderan V. Un peso alto no demuestra una causa ni que la fuente sea correcta.

**Registra en tu copia:** qué representan las filas, qué valores cambiaste, qué observaste y qué límite tiene la conclusión. No es una entrega ni una calificación.

[Volver al tema del aula](https://kthyon.github.io/diplomado-utp-aula/#tema/dl-atencion)


## Tema 3 · Comprobar el orden con un Transformer

**Lo que vas a poder hacer:** Construir una prueba de clasificación de secuencias con respuestas verificables.

Una arquitectura solo puede evaluarse después de definir qué debería responder.

### Entiende la idea

Un Transformer es una forma de organizar una red para trabajar con secuencias. La atención permite que cada parte combine información de otras partes. La posición informa en qué lugar aparece cada entrada: sin ella se puede confundir el primer valor con el último. Cada entrada se expresa como una lista de números para que la red pueda calcular; a esa lista se le llama representación vectorial. No hace falta empezar por sus matrices: primero resolveremos una pregunta de cuatro entregas y sabremos cuál debe ser la respuesta correcta. Después comprobaremos qué calculó la red.

### Un caso pequeño antes del programa

**¿El primero es mayor que el último?**

Cada fila contiene cajas recibidas en cuatro horas ficticias. La etiqueta 1 significa que la última cantidad es menor que la primera; en caso contrario, 0. Es una comparación de extremos, no una afirmación de que todas las horas hayan bajado ni un pronóstico de demanda. Las cantidades entre 1 y 7 se eligieron para poder calcular las etiquetas a mano.

| Secuencia | Primero | Último | Etiqueta |
| --- | --- | --- | --- |
| 1, 2, 3, 4 | 1 | 4 | 0 |
| 4, 3, 2, 1 | 4 | 1 | 1 |
| 2, 7, 1, 3 | 2 | 3 | 0 |
| 3, 1, 7, 2 | 3 | 2 | 1 |

1. Compara extremos: 1 < 4 exige 0; 4 > 1 exige 1.
2. Cambiar el orden de los mismos números puede cambiar la etiqueta.
3. La red debe aprender esta regla. La etiqueta calculada no es su predicción.

**Qué demuestra:** La posición es necesaria para esta regla: [1,2,3,4] requiere 0 y [4,3,2,1] requiere 1. La tabla da respuestas esperadas; para saber qué predice el Transformer hay que ejecutar la red y leer su evaluación.

### Ahora identifica los datos de esta práctica

Compara respuesta esperada y calculada para las secuencias pequeñas. La pregunta es si el último número es menor que el primero.

El caso pequeño y la ejecución pueden usar conjuntos diferentes: interpreta las cifras con las unidades y filas declaradas en cada uno.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere ejecutar los temas anteriores. Si repites un tema, comienza desde su primera celda.

**Qué cambiar y qué conservar:** En 3→1→7→2 la respuesta es sí, aunque la secuencia no baje todo el tiempo. No cambies la pregunta al interpretar el resultado.


In [ ]:
# Preparación 1: instalar únicamente bibliotecas que falten.
import importlib.util, subprocess, sys
paquetes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
            'sklearn':'scikit-learn'}
for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                           '--index-url', 'https://download.pytorch.org/whl/cpu'])
print('Bibliotecas listas. No necesitas activar GPU.')

# Las extensiones se preparan solo cuando se abre su bloque.
def preparar_extension(modulo, paquete):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])


In [ ]:
# Preparación 2: herramientas compartidas.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
from sklearn.datasets import load_digits
from IPython.display import display

# Estas opciones mantienen fuera de la ruta principal el material de ampliación.
# Déjalas en False para los cuatro encuentros de DL.
EJECUTAR_REPASO_ML = False
EJECUTAR_TEXTO = False
EJECUTAR_GENERACION = False

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA); torch.set_num_threads(1)
AZUL, VERDE, GRIS = '#0075ad', '#198a87', '#9eb4c7'
plt.rcParams.update({'figure.figsize':(9,4.5), 'font.size':11,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.titleweight':'bold', 'axes.grid':False})
# En Colab puedes descargar los PNG desde el panel Archivos.
CARPETA = Path('figuras_utp'); CARPETA.mkdir(exist_ok=True)
def mostrar(nombre):
    plt.tight_layout()
    plt.savefig(CARPETA / (nombre + '.png'), dpi=150, bbox_inches='tight')
    plt.show()

def datos_fincas(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Humedad_pct':r.uniform(20,70,n),
                      'Temperatura_C':r.uniform(18,35,n),
                      'Lluvia_mm':r.uniform(0,100,n)})
    y = (2 + .07*X.Humedad_pct + .012*X.Lluvia_mm
         - .025*(X.Temperatura_C-24)**2 + r.normal(0,.3,n))
    return X, y.to_numpy()

def datos_cafe(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Uniformidad_color':r.uniform(0,1,n),
                      'Proporcion_manchas':r.uniform(0,1,n)})
    puntuacion = 2*X.Uniformidad_color-X.Proporcion_manchas-.5+r.normal(0,.15,n)
    return X, (puntuacion > 0).astype(int).to_numpy() # 1=sano, 0=defectuoso
print('Semilla:',SEMILLA,'· PyTorch:',torch.__version__)
print('Todas las cifras agrícolas son simuladas.')


### Consulta, claves y valores son símbolos

Estas palabras ya aparecen en esta práctica. El nombre completo ayuda a leer el código; no añade una tarea nueva.

- **Q, K y V** — Q, K y V: Consulta, clave y valor. Q identifica la consulta; K, las claves con las que se compara; V, la información que se mezcla. Son símbolos de cantidades, no tres siglas de algoritmos. La comparación Q–K determina pesos; la combinación de V produce la salida, sin demostrar causalidad. [Fuente](https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html)

- **Softmax** — Softmax: Transformación de puntuaciones en participaciones. Para cada puntuación calcula su exponencial y la divide entre la suma de exponenciales del grupo. Así obtiene valores positivos que suman uno cuando las entradas son finitas. La normalización se hace sobre un eje elegido; en atención, por las fuentes de cada consulta. [Fuente](https://docs.pytorch.org/docs/stable/generated/torch.nn.Softmax.html)

- **Transformer** — Transformer: Arquitectura de red basada en atención. Arquitectura que combina atención con otras transformaciones, conexiones residuales y normalización. Incorpora información de posición para distinguir órdenes en la tarea del cuaderno. No es una sigla, una sola operación ni necesariamente un chatbot. [Fuente](https://arxiv.org/abs/1706.03762)


### Lee una mezcla antes de leer sus símbolos

Si tomas la mitad de 10 y la mitad de 20, tienes 5+10=15. En la cuenta de atención, una consulta [1,0] da aproximadamente 67 % de participación a A y 33 % a B: la mezcla queda cerca de 13,3, más próxima a 10.

**Vector** significa lista ordenada de números. [1,0] tiene dos posiciones; no es una fracción. **Producto punto** multiplica números del mismo lugar y suma: [1,0] con [1,0] da 1×1 + 0×0 = 1. Se divide por la raíz de 2 porque las listas tienen dos posiciones. **Softmax** convierte esas comparaciones en participaciones positivas que suman 1.

**Tip:** la consulta [0,0] compara igual con A y B. Comprueba que su reparto sea 0,5 y 0,5 y que la salida sea 15. Atención calcula una representación; 13,3 no es una orden de compra ni una cantidad aprendida de ventas.

En el Transformer siguiente sí se ajustan parámetros usando ejemplos de una tarea distinta. Compara sus respuestas con las cuatro etiquetas que resolviste. **Etiqueta** es la respuesta correcta conocida; **predicción** es lo que calcula la red. Una puntuación alta de la red todavía puede equivocarse.


### 12A · Mezclar dos registros con atención

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


In [ ]:
# Dos registros como en el aula: mezclamos cantidades con pesos dados por la comparación.
fuentes=['A: 10','B: 20']
consultas=['Q=[1,0]','Q=[0,1]','Q=[0,0]']
Q=np.array([[1.,0.],[0.,1.],[0.,0.]])
K=np.array([[1.,0.],[0.,1.]])
V=np.array([[10.],[20.]])
puntuaciones=Q@K.T/np.sqrt(2)
exp=np.exp(puntuaciones-puntuaciones.max(axis=1,keepdims=True))
atencion=exp/exp.sum(axis=1,keepdims=True)
mezclas=atencion@V
assert np.allclose(atencion.sum(axis=1),1)
assert np.allclose(atencion[2],[.5,.5]) and np.isclose(mezclas[2,0],15)
for consulta,pesos,mezcla in zip(consultas,atencion,mezclas[:,0]):
    print(consulta,'· participación A/B:',np.round(pesos,4),'· mezcla:',round(float(mezcla),3))
fig,axs=plt.subplots(1,2,figsize=(10,4))
axs[0].imshow(atencion,cmap='Blues',vmin=0,vmax=1)
axs[0].set_xticks(range(2),fuentes);axs[0].set_yticks(range(3),consultas)
for i in range(3):
    for j in range(2):axs[0].text(j,i,f'{atencion[i,j]*100:.1f} %',ha='center',va='center',color='black')
axs[0].set(title='Cuánto participa cada fuente',xlabel='Fuente',ylabel='Consulta dada')
axs[1].bar(consultas,mezclas[:,0],color=[AZUL,VERDE,GRIS]);axs[1].set(title='Resultado de cada mezcla',ylabel='Valor mezclado',ylim=(0,22))
axs[1].axhline(10,ls=':',color='gray');axs[1].axhline(20,ls=':',color='gray')
mostrar('12-atencion')


### Después de ejecutar: comprueba el reparto

Cada fila debe sumar 100 %. Q=[0,0] produce partes iguales y mezcla 15. Q=[1,0] da más participación a A=10 y una salida cercana a 13,3. No se entrenó atención aquí: Q, K y V fueron elegidos para enseñar la operación.


### 12B · Comprobar el orden con un Transformer

Ejecuta la celda siguiente después de leer la explicación del bloque. Espera a que termine antes de interpretar la salida.


In [ ]:
# Transformer real, pequeño, entrenado para una tarea de orden.
r=np.random.default_rng(SEMILLA);seq=r.integers(1,8,size=(700,4));seq=seq[seq[:,0]!=seq[:,-1]]
seq=np.unique(seq,axis=0) # reserva de patrones distintos, no solo de filas
y=(seq[:,0]>seq[:,-1]).astype('int64')
Xtr,Xte,ytr,yte=train_test_split(seq,y,test_size=.25,stratify=y,random_state=SEMILLA)
compartidas=set(map(tuple,Xtr)) & set(map(tuple,Xte))
assert not compartidas, 'Entrenamiento y prueba comparten una secuencia.'
print('Secuencias distintas:',len(seq),'· entrenamiento:',len(Xtr),'· prueba:',len(Xte))
print('Clases [0, 1] en entrenamiento:',np.bincount(ytr),'· en prueba:',np.bincount(yte))
class OrdenTransformer(nn.Module):
    def __init__(self):
        super().__init__();self.emb=nn.Embedding(8,16);self.pos=nn.Parameter(torch.randn(1,4,16)*.1)
        capa=nn.TransformerEncoderLayer(d_model=16,nhead=2,dim_feedforward=32,dropout=0,batch_first=True)
        self.encoder=nn.TransformerEncoder(capa,num_layers=1,enable_nested_tensor=False);self.salida=nn.Linear(4*16,2)
    def forward(self,x):
        # La salida por posiciones también conserva orden; no aislamos el efecto de self.pos.
        h=self.encoder(self.emb(x)+self.pos);return self.salida(h.flatten(1))
torch.manual_seed(SEMILLA);transformer=OrdenTransformer();opt=torch.optim.Adam(transformer.parameters(),lr=.005)
loader=DataLoader(TensorDataset(torch.tensor(Xtr),torch.tensor(ytr)),batch_size=64,shuffle=True)
hist=[]
for epoca in range(45):
    transformer.train();suma=0
    for bx,by in loader:
        opt.zero_grad();error=nn.functional.cross_entropy(transformer(bx),by);error.backward();opt.step();suma+=error.item()*len(bx)
    hist.append(suma/len(Xtr))
transformer.eval()
with torch.no_grad():
    pred=transformer(torch.tensor(Xte)).argmax(1).numpy()
    ejemplos=torch.tensor([[1,2,3,4],[4,3,2,1],[2,7,1,3],[3,1,7,2]]);prob=torch.softmax(transformer(ejemplos),dim=1).numpy()
print('Accuracy en prueba:',round(accuracy_score(yte,pred),3))
esperadas=(ejemplos[:,0]>ejemplos[:,-1]).numpy().astype(int)
calculadas=prob.argmax(axis=1)
for secuencia,esperada,calculada,p1 in zip(ejemplos.tolist(),esperadas,calculadas,prob[:,1]):
    print(secuencia,'· respuesta conocida:',int(esperada),'· respuesta de la red:',int(calculada),'· puntuación clase 1:',round(float(p1),3))
print('Comparar extremos no pregunta si toda la secuencia baja. Las respuestas de la red pueden fallar.')
plt.plot(hist,color=AZUL);plt.xlabel('Época');plt.ylabel('Pérdida');plt.title('Pérdida de entrenamiento en la tarea de orden');mostrar('12-transformer')


### Después de ejecutar: respuesta correcta y calculada

La etiqueta de los cuatro ejemplos se puede comprobar mirando solo primero y último. La red se comprueba contra esa etiqueta; si falla, el resultado sigue siendo un error aunque su puntuación sea alta. El porcentaje de prueba resume otros patrones reservados y no certifica tareas de lenguaje.


### Lee, comprueba y explica

Compara respuesta esperada y calculada para las secuencias pequeñas. La pregunta es si el último número es menor que el primero.

**Prueba pequeña:** En 3→1→7→2 la respuesta es sí, aunque la secuencia no baje todo el tiempo. No cambies la pregunta al interpretar el resultado.

**Vuelve al caso:** Define la regla de respuesta y conserva el orden de los elementos. Comprueba las predicciones con etiquetas conocidas de casos reservados.

**Pregunta:** ¿Un Transformer es lo mismo que atención o que un chatbot?

**Respuesta razonada:** No. Atención es una operación dentro de la arquitectura. Un Transformer añade representación de entrada, posición y otras transformaciones. Puede resolver distintas tareas; el ejercicio clasifica secuencias numéricas y no genera conversación.

**Registra en tu copia:** qué representan las filas, qué valores cambiaste, qué observaste y qué límite tiene la conclusión. No es una entrega ni una calificación.

[Volver al tema del aula](https://kthyon.github.io/diplomado-utp-aula/#tema/dl-transformers)


## Cierra el encuentro con una explicación

1. **El orden puede cambiar el significado:** Conserva el orden y registra cada actualización del estado. El estado resume lo leído hasta ese paso.
2. **Relacionar elementos con contexto:** Sigue consulta y claves → puntuaciones → pesos → suma ponderada de valores. Los pesos indican cuánto aporta cada valor a esa combinación.
3. **Atención más información de orden:** Define la regla de respuesta y conserva el orden de los elementos. Comprueba las predicciones con etiquetas conocidas de casos reservados.

En cada resultado identifica entrada, salida, unidad o significado, forma de comprobarlo y límite. Guarda tu copia personal. Colab no registra automáticamente tu avance en el aula.
